# Testing the `Character` class

## Setup

In [ ]:
import notebook_setup
%preparse gaknot
from gaknot.gaknot import GeneralizedAlgebraicKnot
%preparse H1_branched_cover
from gaknot.H1_branched_cover import BranchedCoverHomology
%preparse character
from gaknot.character import Character
from sage.all import QQ, ZZ
import ipytest
import pytest

ipytest.autoconfig(raise_on_error=True)

## Tests

### Test Case 1: Simple Torus Knots (Parametric - 10 Cases)

**Subject:** Simple Torus Knots $T(p,q)$ at varying covering degrees $N$.

This test verifies that the `Character` class correctly handles simple torus knots $T(p,q)$ for various values of $p, q$ and covering degrees $N$. It checks that the homology factors match the expected ones and that a character can be correctly defined and retrieved using `restrict_to_layer`.

In [ ]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("p, q, N, expected_factors, char_values", [
    (2, 3, 2, [3],       [1/3]),          # 1. Trefoil (Z/3)
    (2, 5, 2, [5],       [1/5]),          # 2. T(2,5) (Z/5)
    (2, 7, 2, [7],       [1/7]),          # 3. T(2,7) (Z/7)
    (2, 3, 3, [2, 2],    [1/2, 1/2]),     # 4. T(2,3) at N=3 (Z/2 + Z/2)
    (3, 4, 2, [3],       [1/3]),          # 5. T(3,4) (Z/3)
    (3, 5, 2, [],        []),             # 6. T(3,5) (Trivial H1)
    (2, 5, 4, [5],       [1/5]),          # 7. T(2,5) at N=4 (Z/5) - Corrected
    (2, 11, 2, [11],     [1/11]),         # 8. T(2,11) (Z/11)
    (2, 31, 2, [31],     [1/31]),         # 9. T(2,31) (Z/31)
    (3, 2, 4, [3],       [1/3])           # 10. T(3,2) at N=4 (Z/3) - Corrected
])
def test_case_1_simple_torus_parametric(p, q, N, expected_factors, char_values):
    # 1. Setup Knot
    knot = GeneralizedAlgebraicKnot([(1, [(p, q)])])
    homology = BranchedCoverHomology(knot, N)
    
    # 2. Verify Homology Factors
    actual_factors = [f for f in homology.invariant_factors if f > 1]
    assert actual_factors == expected_factors
    
    # 3. Define Character
    nested_values = [ [char_values] ]
    chi = Character(homology, nested_values)
    
    # 4. Assertions
    layer_vals = chi.restrict_to_layer(0, 0)
    assert layer_vals == [char_values]

### Test Case 2: Degenerate Satellite (Trivial Outer Layer)

**Subject:** Satellite Knots where one layer contributes no generators.

This test handles "degenerate" satellites where one of the layers (usually the outer pattern layer) has trivial homology. It ensures that the `Character` class correctly identifies the layers and allows assigning character values to the non-trivial ones while respecting the structure of the homology decomposition.

In [ ]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("cable, N, outer_factors, inner_factors, inner_values", [
    ([(2, 3), (2, 5)], 3, [], [2, 2], [1/2, 1/2]),
    ([(2, 5), (3, 5)], 2, [], [5], [1/5]),
    ([(3, 4), (3, 5)], 2, [], [3], [1/3]),
    ([(2, 3), (3, 7)], 2, [], [3], [1/3]),
    ([(2, 11), (2, 3)], 3, [2, 2], [], []),
    ([(2, 3), (2, 5)], 5, [2, 2, 2, 2], [], []),
    ([(3, 2), (2, 3)], 2, [3], [], []),
    ([(2, 5), (2, 7)], 2, [7], [], []),
    ([(2, 3), (2, 11)], 2, [11], [], []),
    ([(2, 7), (2, 5)], 3, [], [], [])
])
def test_case_2_degenerate_satellite(cable, N, outer_factors, inner_factors, inner_values):
    knot = GeneralizedAlgebraicKnot([(1, cable)])
    homology = BranchedCoverHomology(knot, N)
    
    # Verify structure matches expectations
    # layers[0] is outer, layers[1] is inner
    assert homology.decomposition[0]["layers"][0]["base_factors"] == outer_factors
    assert homology.decomposition[0]["layers"][1]["base_factors"] == inner_factors
    
    # Define Character
    # Note: Multiplicity check
    m_outer = homology.decomposition[0]["layers"][0]["multiplicity"]
    m_inner = homology.decomposition[0]["layers"][1]["multiplicity"]
    
    outer_vals_input = [0] * (m_outer * len(outer_factors))
    inner_vals_input = inner_values # Assumed to be provided with multiplicity in mind or simple
    # Fix for multiplicity: if inner_values is just factors, repeat it
    if len(inner_vals_input) != m_inner * len(inner_factors):
        inner_vals_input = inner_values * m_inner
        
    values = [ [ outer_vals_input, inner_vals_input ] ]
    chi = Character(homology, values)
    
    # Assertions
    assert chi.restrict_to_layer(0, 0) == [ [0]*len(outer_factors) ] * m_outer
    assert chi.restrict_to_layer(0, 1) == [ inner_values ] * m_inner


### Test Case 3: Fully Nontrivial Satellite

**Subject:** Satellite Knots where both layers contribute generators.

This test examines satellite knots where both the companion and the pattern layers contribute non-trivial homology factors. It verifies that characters can be assigned to all generators across both layers and that `restrict_to_layer` correctly extracts these values.

In [ ]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("cable, N, outer_factors, inner_factors", [
    ([(2, 5), (3, 2)], 2, [3], [5]),
    ([(2, 3), (3, 2)], 2, [3], [3]),
    ([(2, 3), (5, 2)], 2, [5], [3]),
    ([(2, 7), (3, 2)], 2, [3], [7]),
    ([(2, 5), (5, 2)], 2, [5], [5]),
    ([(2, 3), (7, 2)], 2, [7], [3]),
    ([(2, 11), (3, 2)], 2, [3], [11]),
    ([(3, 2), (5, 2)], 2, [5], [3]),
    ([(5, 2), (3, 2)], 2, [3], [5]),
    ([(2, 5), (7, 2)], 2, [7], [5])
])
def test_case_3_nontrivial_cable_full(cable, N, outer_factors, inner_factors):
    knot = GeneralizedAlgebraicKnot([(1, cable)])
    homology = BranchedCoverHomology(knot, N)
    
    # Map each generator to 1/modulus
    o_vals = [1/f for f in outer_factors]
    i_vals = [1/f for f in inner_factors]
    
    values = [ [ o_vals, i_vals ] ]
    chi = Character(homology, values)
    
    assert chi.restrict_to_layer(0, 0) == [o_vals]
    assert chi.restrict_to_layer(0, 1) == [i_vals]


### Test Case 4: Connected Sum (Geometric vs. Sorted Order)

This test confirms that the `Character` class assigns values based on the "Geometric Order" of components (the order in which they are added in the connected sum) rather than the sorted order of Smith Normal Form factors. This is crucial for maintaining a 1-to-1 correspondence with the knot's recursive construction.

In [ ]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("p1, q1, p2, q2, N", [
    (5, 2, 3, 2, 2),
    (7, 2, 3, 2, 2),
    (11, 2, 5, 2, 2),
    (3, 2, 5, 2, 2),
    (5, 2, 7, 2, 2),
    (2, 3, 2, 5, 3),
    (2, 5, 2, 3, 2),
    (3, 4, 2, 3, 2),
    (2, 3, 3, 4, 2),
    (2, 11, 2, 3, 2)
])
def test_case_4_connected_sum_geometric_order(p1, q1, p2, q2, N):
    k1 = GeneralizedAlgebraicKnot([(1, [(p1, q1)])])
    k2 = GeneralizedAlgebraicKnot([(1, [(p2, q2)])])
    knot_sum = k1 + k2
    homology = BranchedCoverHomology(knot_sum, N)
    
    f1 = [f for f in BranchedCoverHomology(k1, N).invariant_factors if f > 1]
    f2 = [f for f in BranchedCoverHomology(k2, N).invariant_factors if f > 1]
    
    v1 = [1/f for f in f1]
    v2 = [1/f for f in f2]
    
    values = [ [v1], [v2] ]
    chi = Character(homology, values)
    
    assert chi.restrict_to_layer(0, 0) == [v1]
    assert chi.restrict_to_layer(1, 0) == [v2]

### Test Case 5: Modulus Compatibility Validation

This test verifies the mathematical validation logic. It ensures that the `Character` class rejects rational values that are incompatible with the order of the corresponding homology generator (i.e., $v \cdot m$ must be an integer, where $m$ is the modulus).

In [ ]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("p, q, N, bad_val", [
    (2, 3, 3, 1/3), # H1 = Z/2 + Z/2
    (2, 5, 2, 1/3), # H1 = Z/5
    (2, 3, 2, 1/2), # H1 = Z/3
    (3, 4, 2, 1/2), # H1 = Z/3
    (2, 7, 2, 1/5), # H1 = Z/7
    (2, 5, 4, 1/2), # H1 = Z/5
    (2, 11, 2, 1/3), # H1 = Z/11
    (2, 3, 3, 1/7), # H1 = Z/2 + Z/2
    (3, 2, 4, 1/2), # H1 = Z/3
    (2, 31, 2, 1/2)  # H1 = Z/31
])
def test_validation_modulus_compatibility(p, q, N, bad_val):
    knot = GeneralizedAlgebraicKnot([(1, [(p, q)])])
    homology = BranchedCoverHomology(knot, N)
    factors = [f for f in homology.invariant_factors if f > 1]
    
    if not factors:
        pytest.skip("Trivial homology")
        
    bad_values = [ [ [bad_val] * len(factors) ] ]
    with pytest.raises(ValueError, match="not compatible with Z"):
        Character(homology, bad_values)

### Test Case 6: Heterogeneous Connected Sum (Simple + Satellite)

This test handles "jagged" structures combining simple torus knots and satellite knots. It ensures that characters are correctly assigned and extracted when the components have different hierarchical depths and multiplicities.

In [ ]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("simple_pq, satellite_cable, N", [
    ((2, 5), [(2, 5), (2, 3)], 2),
    ((2, 3), [(2, 3), (2, 5)], 2),
    ((3, 2), [(2, 5), (3, 2)], 2),
    ((2, 7), [(2, 3), (2, 7)], 2),
    ((2, 3), [(2, 5), (2, 3)], 2),
    ((2, 5), [(2, 7), (2, 5)], 2),
    ((2, 11), [(2, 3), (2, 11)], 2),
    ((3, 4), [(2, 3), (3, 4)], 2),
    ((2, 5), [(3, 2), (2, 5)], 2),
    ((2, 3), [(2, 11), (2, 3)], 2)
])
def test_heterogeneous_connected_sum(simple_pq, satellite_cable, N):
    k1 = GeneralizedAlgebraicKnot([(1, [simple_pq])])
    k2 = GeneralizedAlgebraicKnot([(1, satellite_cable)])
    knot_sum = k1 + k2
    homology = BranchedCoverHomology(knot_sum, N)
    
    # Get actual factors for values assignment
    h1 = BranchedCoverHomology(k1, N)
    h2 = BranchedCoverHomology(k2, N)
    
    v1 = [1/f for f in h1.invariant_factors if f > 1]
    
    # Layer-wise values for satellite
    v_sat = []
    for layer in h2.decomposition[0]['layers']:
        m = layer['multiplicity']
        f = layer['base_factors']
        v_sat.append([1/x for x in f] * m)
        
    values = [ [v1], v_sat ]
    chi = Character(homology, values)
    
    assert chi.restrict_to_layer(0, 0) == [v1] if v1 else [[]]
    for i, layer_vals in enumerate(chi.restrict_to_layer(1, i) for i in range(len(v_sat))):
        expected = [ [1/x for x in h2.decomposition[0]['layers'][i]['base_factors']] ] * h2.decomposition[0]['layers'][i]['multiplicity']
        assert layer_vals == expected

### Test Case 7: Validation of Mixed Validity (Satellite Layers)

This test confirms that validation iterates correctly through satellite layers. It checks a case where the outer layer values are valid but the inner layer values are mathematically incompatible, ensuring the class catches errors at any depth.

In [ ]:
%%ipytest -vv -W ignore::DeprecationWarning

def test_validation_mixed_validity_satellite():
    # Case: Outer layer valid, inner layer invalid
    # T(2,3; 3,2) at N=2: Outer H1=Z/3 (mult=1), Inner H1=Z/3 (mult=1)
    knot = GeneralizedAlgebraicKnot([(1, [(2, 3), (3, 2)])])
    homology = BranchedCoverHomology(knot, 2)
    
    # 1. Valid Character
    valid_values = [ [ [1/3], [1/3] ] ]
    Character(homology, valid_values)
    
    # 2. Invalid Inner Layer (Z/3 generator given 1/2)
    invalid_values = [ [ [1/3], [1/2] ] ]
    with pytest.raises(ValueError, match="not compatible with Z/3Z"):
        Character(homology, invalid_values)

### Test Case 8: Heterogeneous Connected Sum (Nontrivial Inner Cover)

This test complements Case 6 by verifying cases where the inner companion of a satellite component preserves non-trivial homology even when the branching degree splits. It ensures correct mapping across multiple copies of the inner cover.

In [ ]:
%%ipytest -vv -W ignore::DeprecationWarning

def test_heterogeneous_connected_sum_nontrivial_inner():
    # K = T(2,3) # T(2,5; 3,2) at N=2
    # T(2,3) at N=2: H1 = Z/3
    # T(2,5; 3,2) at N=2:
    #   Outer: T(3,2) at N=2: H1 = Z/3
    #   Inner: T(2,5) at N=2: H1 = Z/5 (mult=1)
    
    k1 = GeneralizedAlgebraicKnot([(1, [(2, 3)])])
    k2 = GeneralizedAlgebraicKnot([(1, [(2, 5), (3, 2)])])
    knot = k1 + k2
    homology = BranchedCoverHomology(knot, 2)
    
    # Expected Structure:
    # Comp 0: [[1/3]]
    # Comp 1: [[1/3], [1/5]]
    values = [
        [[1/3]],
        [[1/3], [1/5]]
    ]
    chi = Character(homology, values)
    
    assert chi.restrict_to_layer(0, 0) == [[1/3]]
    assert chi.restrict_to_layer(1, 0) == [[1/3]]
    assert chi.restrict_to_layer(1, 1) == [[1/5]]

### Test Case 9: Complex Sum ($K_3$ at $N=3$)

This test automates the complex case of $K_3 = T(2,3; 2,9; 2,21) \# -T(5,6; 5,12; 7,15)$ at $N=3$. It performs an exhaustive structural check on a deeply nested and heterogeneous knot sum, ensuring the `Character` class maintains integrity even for complex real-world examples.

In [ ]:
%%ipytest -vv -W ignore::DeprecationWarning

def test_complex_k3_case():
    # K3 = T(2,3; 2,9; 2,21) # -T(5,6; 5,12; 7,15) at N=3
    desc1 = [(2, 3), (2, 9), (2, 21)]
    desc2 = [(5, 6), (5, 12), (7, 15)]
    knot = GeneralizedAlgebraicKnot([(1, desc1), (-1, desc2)])
    homology = BranchedCoverHomology(knot, 3)
    
    # Map all generators to 1/modulus
    values = []
    for comp in homology.decomposition:
        comp_vals = []
        for layer in comp['layers']:
            m = layer['multiplicity']
            f = layer['base_factors']
            comp_vals.append([1/x if x != 0 else 0 for x in f] * m)
        values.append(comp_vals)
        
    chi = Character(homology, values)
    
    # Basic integrity check
    assert len(chi.values) == len(homology.invariant_factors)
    for i, comp in enumerate(homology.decomposition):
        for j, layer in enumerate(comp['layers']):
            m = layer['multiplicity']
            layer_vals = chi.restrict_to_layer(i, j)
            assert len(layer_vals) == m
            for copy_vals in layer_vals:
                assert len(copy_vals) == len(layer['base_factors'])